In [0]:
import pyspark.sql.functions as F

df = spark.table("workspace.erp_lakehouse.bronze_b_items")

# change all the uom value to a uppoer case. 
df = df.withColumn("uom", F.upper(F.col("uom")))
df.display()


In [0]:
# Data cleaning
# Update EACH to EA since they mean the same thing. This will prevent issues when grouping data.
#
df = df.withColumn("uom", F.when(df.uom == 'EACH', 'EA').otherwise(df.uom))
df.display()


In [0]:
# casing the list_price the data need to be change to a desimal to do math
# Data cleaning
# Update the data to be in the correct format
print(df.schema['list_price'].dataType)
df = df.withColumn("list_price", F.expr("try_cast(list_price AS DECIMAL(10,2))"))
df.display()
print(f"after update {df.schema['list_price'].dataType}")

In [0]:
# Remove stray spaces so the same value isn't treated as two different values
df = (df
    .withColumn("part_no", F.trim(F.col("part_no")))
    .withColumn("uom", F.trim(F.col("uom"))))

In [0]:
from pyspark.sql.window import Window

# Same part_no twice: keep the row with the higher price
w = Window.partitionBy("part_no").orderBy(F.col("list_price").desc())
items_silver = (df
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn"))
items_silver.display()

In [0]:
items_silver.groupBy("uom").count().display()   # only EA should show
print(items_silver.count(), items_silver.select("part_no").distinct().count())   # both numbers must match

In [0]:
items_silver.write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.erp_lakehouse.silver_b_items")